# 02 — Entrenamiento de los detectores

Entrena **10 modelos**: TranAD y TranAD+ con las semillas **0, 7, 42, 123 y 777**, fijadas antes de ver ningún resultado (apartado 4.3.1 de la memoria).

| Modelo | `dim_feedforward` | Puntuación (α, β) |
|---|---|---|
| TranAD | 16 | (0, 1): solo el segundo decodificador |
| TranAD+ | 64 (cuello de botella inverso) | (0,5; 0,5): los dos decodificadores |

**Qué se guarda por cada modelo** (en `RUTA_DETECTORES`):
- `checkpoints/<modelo>_s<semilla>/`: un checkpoint por época, con el estado aleatorio, para poder reanudar.
- `errores/<modelo>_s<semilla>.npz`: error de reconstrucción por variable de cada decodificador por separado, para cada ventana de `val`, `test` y `valores_imposibles`. Con ellos se calcula cualquier puntuación sin volver a usar la GPU. El orden de las ventanas es el de `ventanas_escaladas.npz`.
- `registro_entrenamientos.csv`: una fila por modelo.

**Duración:** unos 27 minutos por modelo en una GPU Tesla T4 (unas 4,5 horas en total).

**Si ya existen los checkpoints** de la época 15 en `RUTA_DETECTORES/checkpoints`, no se reentrena: se cargan y solo se recalculan los errores (unos minutos por modelo).

**Si Colab se corta:** volver a ejecutar las secciones 1 a 4 y después la celda de la semilla interrumpida. Lo terminado se salta y el modelo interrumpido continúa desde su último checkpoint.

**Antes de ejecutar:** reiniciar el entorno de Colab, para que la configuración determinista de la GPU tenga efecto.

La sección 7 (opcional) repite los 10 entrenamientos en otra carpeta y comprueba que se obtienen los mismos modelos.

## 1. Entorno

En Colab, monta Google Drive y clona el repositorio para tener acceso a `src/`. En local, busca la carpeta `src/` subiendo desde el directorio actual.

In [ ]:
# --- Entorno: localiza src/ y, en Colab, monta Google Drive ---
import os
import subprocess
import sys

# Debe definirse antes de cualquier operación en GPU para que cuBLAS sea determinista.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/Williams-MP/TFM_VIU_2026.git"
RUTA_REPO_COLAB = "/content/TFM_VIU_2026"

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(RUTA_REPO_COLAB):
        # Mientras el repositorio sea privado hace falta un token de GitHub.
        # Si es público, basta con dejar el campo vacío.
        from getpass import getpass
        token = getpass("Token de GitHub (vacío si el repositorio es público): ")
        url = URL_REPOSITORIO.replace("https://", f"https://{token}@") if token else URL_REPOSITORIO
        subprocess.run(["git", "clone", "-q", url, RUTA_REPO_COLAB], check=True)
    ruta_src = f"{RUTA_REPO_COLAB}/src"
else:
    # Ejecución local: se sube por las carpetas hasta encontrar src/.
    directorio = os.getcwd()
    while not os.path.isdir(os.path.join(directorio, "src")):
        padre = os.path.dirname(directorio)
        if padre == directorio:
            raise FileNotFoundError("No se encuentra la carpeta src/ del repositorio.")
        directorio = padre
    ruta_src = os.path.join(directorio, "src")

if ruta_src not in sys.path:
    sys.path.append(ruta_src)

import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from config import RUTA_PREPARADOS, RUTA_DETECTORES, N_FEATS, N_WINDOW
from entrenamiento import entrenar_modelo
from evaluacion import calcular_anomaly_scores, calcular_errores_por_senal, score_desde_errores, umbral_pot

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NOMBRE_GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DISPOSITIVO, "|", NOMBRE_GPU, "| PyTorch:", torch.__version__)
print("Resultados en:", RUTA_DETECTORES)

## 2. Datos

Se cargan las ventanas preparadas en `01_preparacion_datos`. Los modelos se entrenan solo con `train`; el resto se usa después para calcular las puntuaciones.

In [ ]:
# --- Carga de las ventanas preparadas y comparación de tamaños con la memoria ---
datos = np.load(f"{RUTA_PREPARADOS}/ventanas_escaladas.npz")
X_train = datos["train"]
CONJUNTOS_EVALUACION = {
    "val": datos["val"],
    "test": datos["test"],
    "valores_imposibles": datos["valores_imposibles"],
}

# Tamaños de la memoria (Tabla 5 y apartado 5.1.4). Con otros datos no coinciden, y es normal.
TAMANOS_MEMORIA = {"train": 562_875, "val": 113_034, "test": 88_126, "valores_imposibles": 885}
tamanos = {"train": len(X_train), **{k: len(v) for k, v in CONJUNTOS_EVALUACION.items()}}
for conjunto, n_memoria in TAMANOS_MEMORIA.items():
    marca = "coincide" if tamanos[conjunto] == n_memoria else "distinto"
    print(f"{conjunto:19s} {tamanos[conjunto]:>9,} ventanas (memoria: {n_memoria:,}) -> {marca}")

## 3. Parámetros

Todo queda fijado aquí, antes de entrenar. Las dos arquitecturas se entrenan con los mismos hiperparámetros, para que la comparación del Experimento 1 aísle el efecto de la arquitectura. Los entrenamientos alternan modelo dentro de cada semilla, para tener siempre parejas completas si se acaba el tiempo.

In [ ]:
# --- Parámetros del experimento, fijados antes de entrenar ---
SEMILLAS = [0, 7, 42, 123, 777]

MODELOS = {
    "tranad":     {"dim_feedforward": 16, "alfa": 0.0, "beta": 1.0},
    "tranadplus": {"dim_feedforward": 64, "alfa": 0.5, "beta": 0.5},
}

N_EPOCAS = 15
TAMANO_BATCH = 128
TASA_APRENDIZAJE = 1e-3

# Orden: tranad_s0, tranadplus_s0, tranad_s7, tranadplus_s7, ...
CORRIDAS = [(semilla, nombre_modelo) for semilla in SEMILLAS for nombre_modelo in MODELOS]
print(f"{len(CORRIDAS)} entrenamientos:", ", ".join(f"{m}_s{s}" for s, m in CORRIDAS))

## 4. Funciones

`ejecutar_corrida()` reúne los pasos de cada entrenamiento:

1. Si su fichero de errores ya existe, está terminado y se salta.
2. Si no, entrena el modelo con su semilla, o lo reanuda desde su último checkpoint.
3. Calcula y guarda los errores por variable de `val`, `test` y `valores_imposibles`.
4. Comprueba que la puntuación calculada a partir de esos errores coincide con la de `calcular_anomaly_scores()`.
5. Añade una fila al registro.

Recibe la carpeta base como parámetro, para usar la misma función en la comprobación de reproducibilidad de la sección 7.

In [ ]:
# --- Funciones de cada entrenamiento ---

def guardar_npz_seguro(ruta_final, **arrays):
    # Escribe en un fichero temporal y lo renombra al final: un corte de Colab
    # no deja nunca un fichero a medias que parezca completo.
    ruta_temporal = ruta_final + ".tmp"
    with open(ruta_temporal, "wb") as f:
        np.savez_compressed(f, **arrays)
    os.replace(ruta_temporal, ruta_final)


def contar_checkpoints(ruta):
    # Épocas ya guardadas en la carpeta: más de 0 indica que el entrenamiento se reanuda.
    if not os.path.isdir(ruta):
        return 0
    return len([f for f in os.listdir(ruta) if f.startswith("checkpoint_epoca_") and f.endswith(".pt")])


def ejecutar_corrida(semilla, nombre_modelo, ruta_base=RUTA_DETECTORES):
    config = MODELOS[nombre_modelo]
    nombre_corrida = f"{nombre_modelo}_s{semilla}"
    ruta_checkpoints = f"{ruta_base}/checkpoints/{nombre_corrida}"
    ruta_errores = f"{ruta_base}/errores/{nombre_corrida}.npz"
    ruta_registro = f"{ruta_base}/registro_entrenamientos.csv"
    os.makedirs(f"{ruta_base}/errores", exist_ok=True)

    print("=" * 80)
    if os.path.exists(ruta_errores):
        print(f"{nombre_corrida}: ya terminado, se salta.")
        return

    # 1. Entrenamiento (o reanudación). Se registran los avisos de operaciones no deterministas.
    epocas_previas = contar_checkpoints(ruta_checkpoints)
    inicio = time.time()
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        modelo = entrenar_modelo(
            ruta_checkpoints=ruta_checkpoints, dim_feedforward=config["dim_feedforward"],
            X_entrenamiento=X_train, n_feats=N_FEATS, n_window=N_WINDOW, dispositivo=DISPOSITIVO,
            n_epocas=N_EPOCAS, tamano_batch=TAMANO_BATCH, tasa_aprendizaje=TASA_APRENDIZAJE,
            semilla=semilla,
        )
    duracion_entrenamiento = (time.time() - inicio) / 60
    avisos_determinismo = sorted({str(a.message)[:200] for a in avisos if "determinis" in str(a.message).lower()})
    for texto in avisos_determinismo:
        print("Aviso de operación no determinista:", texto)

    perdida_final = torch.load(f"{ruta_checkpoints}/checkpoint_epoca_{N_EPOCAS}.pt",
                               map_location="cpu", weights_only=False)["perdida"]

    # 2. Errores por variable de cada decodificador, en cada conjunto de evaluación.
    inicio = time.time()
    errores = {}
    for nombre_conjunto, X in CONJUNTOS_EVALUACION.items():
        errores[f"{nombre_conjunto}_e1"], errores[f"{nombre_conjunto}_e2"] = calcular_errores_por_senal(
            modelo, X, DISPOSITIVO)

    # 3. La puntuación calculada desde los errores debe coincidir con la directa.
    score_directo = calcular_anomaly_scores(modelo, CONJUNTOS_EVALUACION["test"], DISPOSITIVO,
                                            config["alfa"], config["beta"])
    score_desde_e = score_desde_errores(errores["test_e1"], errores["test_e2"], config["alfa"], config["beta"])
    dif_maxima = float(np.abs(score_directo - score_desde_e).max())
    print(f"Equivalencia de la puntuación en test: diferencia máxima = {dif_maxima:.2e}")
    assert np.allclose(score_directo, score_desde_e, rtol=1e-4, atol=1e-7), \
        "La puntuación calculada desde los errores no coincide: revisar antes de continuar"

    # 4. Guardado, con los parámetros del modelo para que el fichero se explique solo.
    guardar_npz_seguro(ruta_errores, **errores, semilla=np.array(semilla),
                       dim_feedforward=np.array(config["dim_feedforward"]),
                       alfa=np.array(config["alfa"]), beta=np.array(config["beta"]))
    duracion_errores = (time.time() - inicio) / 60

    # 5. Registro.
    fila = pd.DataFrame([{
        "corrida": nombre_corrida, "modelo": nombre_modelo, "semilla": semilla,
        "dim_feedforward": config["dim_feedforward"], "alfa": config["alfa"], "beta": config["beta"],
        "perdida_final": perdida_final,
        "epocas_previas_al_inicio": epocas_previas,  # >0: hubo reanudación (15: no se reentrenó)
        "duracion_entrenamiento_min": round(duracion_entrenamiento, 2),
        "duracion_errores_min": round(duracion_errores, 2),
        "dif_equivalencia_score": dif_maxima,
        "avisos_no_deterministas": len(avisos_determinismo),
        "pytorch": torch.__version__, "gpu": NOMBRE_GPU,
        "fecha": datetime.now().isoformat(timespec="seconds"),
    }])
    fila.to_csv(ruta_registro, mode="a", header=not os.path.exists(ruta_registro), index=False)
    print(f"{nombre_corrida}: terminado | pérdida final = {perdida_final:.6e} | "
          f"entrenamiento = {duracion_entrenamiento:.1f} min | errores = {duracion_errores:.1f} min")

    # Libera la memoria de la GPU antes del siguiente modelo.
    del modelo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Entrenamientos

Una celda por semilla, cada una con TranAD y TranAD+ (unos 55 minutos por celda). Se pueden ejecutar de una en una.

### 5.1 Semilla 0

In [ ]:
# --- Semilla 0: TranAD y TranAD+ ---
for nombre_modelo in MODELOS:
    ejecutar_corrida(0, nombre_modelo)

### 5.2 Semilla 7

In [ ]:
# --- Semilla 7: TranAD y TranAD+ ---
for nombre_modelo in MODELOS:
    ejecutar_corrida(7, nombre_modelo)

### 5.3 Semilla 42

In [ ]:
# --- Semilla 42: TranAD y TranAD+ ---
for nombre_modelo in MODELOS:
    ejecutar_corrida(42, nombre_modelo)

### 5.4 Semilla 123

In [ ]:
# --- Semilla 123: TranAD y TranAD+ ---
for nombre_modelo in MODELOS:
    ejecutar_corrida(123, nombre_modelo)

### 5.5 Semilla 777

In [ ]:
# --- Semilla 777: TranAD y TranAD+ ---
for nombre_modelo in MODELOS:
    ejecutar_corrida(777, nombre_modelo)

## 6. Resumen y comprobación con la memoria

Comprueba que están los 10 ficheros de errores y compara la pérdida de la última época con la Tabla 11 de la memoria. Si una fila del registro aparece repetida (por ejemplo, tras un corte justo después de guardarla), se conserva la última.

In [ ]:
# --- Resumen de los entrenamientos y comparación con la Tabla 11 de la memoria ---
faltan = [f"{m}_s{s}" for s, m in CORRIDAS if not os.path.exists(f"{RUTA_DETECTORES}/errores/{m}_s{s}.npz")]
print(f"Ficheros de errores: {len(CORRIDAS) - len(faltan)}/{len(CORRIDAS)}", f"(faltan: {faltan})" if faltan else "")

# Pérdida de la última época en la memoria (Tabla 11), en unidades de 1e-5.
PERDIDA_MEMORIA = {
    "tranad_s0": 2.11, "tranad_s7": 2.90, "tranad_s42": 1.91, "tranad_s123": 3.06, "tranad_s777": 2.11,
    "tranadplus_s0": 2.06, "tranadplus_s7": 1.96, "tranadplus_s42": 4.28, "tranadplus_s123": 2.75,
    "tranadplus_s777": 1.97,
}

registro = pd.read_csv(f"{RUTA_DETECTORES}/registro_entrenamientos.csv").drop_duplicates(subset="corrida", keep="last")
registro["perdida_final_x1e-5"] = (registro["perdida_final"] * 1e5).round(2)
registro["perdida_memoria_x1e-5"] = registro["corrida"].map(PERDIDA_MEMORIA)
registro["coincide_memoria"] = registro["perdida_final_x1e-5"] == registro["perdida_memoria_x1e-5"]
display(registro[["corrida", "perdida_final_x1e-5", "perdida_memoria_x1e-5", "coincide_memoria",
                  "epocas_previas_al_inicio", "duracion_entrenamiento_min", "dif_equivalencia_score", "gpu"]])

## 7. Comprobación de reproducibilidad (opcional)

Repite los 10 entrenamientos desde cero en otra carpeta y compara cada modelo con el de la sección 5 (apartado 5.1.1 de la memoria). Tarda otras 4,5 horas. Conviene usar la misma GPU y la misma versión de PyTorch: con otras, pueden aparecer diferencias numéricas pequeñas sin que haya ningún error.

| Nivel | Qué se compara | Si coincide |
|---|---|---|
| Pesos | Todos los parámetros en la época 15 | Mismo modelo, bit a bit |
| Pérdidas | Pérdida media de cada una de las 15 épocas | Mismo proceso de entrenamiento |
| Puntuaciones | Errores y puntuación en `val`, `test` y `valores_imposibles` | Mismas salidas |
| Alertas | Umbral POT (sobre `val`) y ventanas de `test` que lo superan | Mismas conclusiones |

**Veredicto:** *idéntico* si los pesos son iguales bit a bit; *equivalente* si no lo son, pero la puntuación difiere como mucho 10⁻⁶ y las alertas son las mismas; *difiere* en cualquier otro caso.

In [ ]:
# --- Repetición de los 10 entrenamientos en una carpeta aparte ---
RUTA_REPRODUCIBILIDAD = f"{RUTA_DETECTORES}_reproducibilidad"
assert RUTA_REPRODUCIBILIDAD != RUTA_DETECTORES  # nunca se escribe sobre los modelos originales

for semilla, nombre_modelo in CORRIDAS:
    ejecutar_corrida(semilla, nombre_modelo, ruta_base=RUTA_REPRODUCIBILIDAD)

In [ ]:
# --- Comparación de cada modelo repetido con el original ---
POT_Q, POT_NIVEL_INICIAL = 1e-3, 0.98  # mismos parámetros que en el apartado 4.3.2


def cargar_checkpoint(ruta_base, nombre_corrida, epoca):
    # Carga un checkpoint en CPU para compararlo.
    return torch.load(f"{ruta_base}/checkpoints/{nombre_corrida}/checkpoint_epoca_{epoca}.pt",
                      map_location="cpu", weights_only=False)


def comparar(nombre_corrida, alfa, beta):
    fila = {"corrida": nombre_corrida}

    # Pesos de la última época.
    p_o = cargar_checkpoint(RUTA_DETECTORES, nombre_corrida, N_EPOCAS)["modelo"]
    p_n = cargar_checkpoint(RUTA_REPRODUCIBILIDAD, nombre_corrida, N_EPOCAS)["modelo"]
    fila["pesos_identicos"] = all(torch.equal(p_o[k], p_n[k]) for k in p_o)
    fila["dif_max_pesos"] = max(float((p_o[k].float() - p_n[k].float()).abs().max()) for k in p_o)

    # Pérdidas época a época: primera época en que dejan de coincidir (si alguna).
    distintas = [e for e in range(1, N_EPOCAS + 1)
                 if cargar_checkpoint(RUTA_DETECTORES, nombre_corrida, e)["perdida"]
                 != cargar_checkpoint(RUTA_REPRODUCIBILIDAD, nombre_corrida, e)["perdida"]]
    fila["primera_epoca_distinta"] = distintas[0] if distintas else None

    # Puntuaciones en todos los conjuntos.
    e_o = np.load(f"{RUTA_DETECTORES}/errores/{nombre_corrida}.npz")
    e_n = np.load(f"{RUTA_REPRODUCIBILIDAD}/errores/{nombre_corrida}.npz")
    fila["dif_max_score"] = max(
        float(np.abs(score_desde_errores(e_o[f"{c}_e1"], e_o[f"{c}_e2"], alfa, beta)
                     - score_desde_errores(e_n[f"{c}_e1"], e_n[f"{c}_e2"], alfa, beta)).max())
        for c in CONJUNTOS_EVALUACION)

    # Alertas en test con el umbral POT de cada versión.
    alertas = []
    for e in (e_o, e_n):
        umbral = umbral_pot(score_desde_errores(e["val_e1"], e["val_e2"], alfa, beta), POT_Q, POT_NIVEL_INICIAL)
        alertas.append(set(np.where(score_desde_errores(e["test_e1"], e["test_e2"], alfa, beta) > umbral)[0]))
    union = alertas[0] | alertas[1]
    fila["alertas_original"], fila["alertas_nuevo"] = len(alertas[0]), len(alertas[1])
    fila["jaccard_alertas"] = len(alertas[0] & alertas[1]) / len(union) if union else 1.0

    if fila["pesos_identicos"]:
        fila["veredicto"] = "idéntico"
    elif fila["dif_max_score"] <= 1e-6 and fila["jaccard_alertas"] == 1.0:
        fila["veredicto"] = "equivalente"
    else:
        fila["veredicto"] = "difiere"
    return fila


tabla = pd.DataFrame([comparar(f"{m}_s{s}", MODELOS[m]["alfa"], MODELOS[m]["beta"]) for s, m in CORRIDAS])
tabla.to_csv(f"{RUTA_REPRODUCIBILIDAD}/comparacion_reproducibilidad.csv", index=False)
display(tabla)
print(tabla["veredicto"].value_counts().to_string())